# Dependencias

In [ ]:
import pandas as pd
import numpy as np

## Initial analysis

In [ ]:
df = pd.read_csv('/home/lucas/Documents/Git/mlops-project/data/Crop_recommendation.csv')

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.info()

Única variável categórica presente é a label. Na versão antiga usei labelencoder, mas desse vez vamos usar pandas nativo e assim colocar o modelo e o standardscaler no mesmo objeto.

# EDA

Já que todos os dados são numéricos isso dispensa a separação de tipos, nos permitindo focar direto em todas as variaveis

## Variável target

In [ ]:
df['label'].describe()

In [ ]:
df['label'].value_counts()

Não há desbalanceamento aqui. As classes são balanceadas com todas possuindo 100 valores para cada classe

## Variaveis independentes

In [ ]:
import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

def univariable_graf(df, variable, log_scale=False):
    sns.set_theme(style="ticks")

    f, ax = plt.subplots(figsize=(12, 5))
    sns.despine(f)

    sns.histplot(
        df,
        x=variable,
        stat='frequency',
        multiple="stack",
        edgecolor=".3",
        linewidth=.5,
        log_scale=log_scale,
    )
    plt.show()
    print(df[variable].describe())
    print('Skewness: ', df[variable].skew())
    print('CV: ', df[variable].std() / df[variable].mean())
    IQR = df[variable].quantile(0.75) - df[variable].quantile(0.25)
    LI = df[variable].quantile(0.25) - (1.5 * IQR)
    LS = df[variable].quantile(0.75) + (1.5 * IQR)
    print('Limite Inferior: ', LI)
    print('Limite Superior: ', LS, '\n')
    




. Encontre o IQR e os LimitesCalcule o IQR:\(IQR = Q_3 - Q_1\)Defina o Limite Inferior:\(Limite\_Inferior = Q_1 - (1.5 \times IQR)\)Defina o Limite Superior:\(Limite\_Superior = Q_3 + (1.5 \times IQR)\)

In [ ]:
df['N'].quantile(0.75)

In [ ]:
for col in df.columns:
    if col != 'label':
        univariable_graf(df, col)

In [ ]:
df['ph'].mode().mean()

## Analise Bivariada 

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme()

# Draw a heatmap with the numeric values in each cell
f, ax = plt.subplots(figsize=(9, 6))
sns.heatmap(df.drop('label', axis=1).corr(), annot=True, linewidths=.5, ax=ax)

In [ ]:
# Filtrando a matriz de correlação
dfCorr = df.drop('label', axis=1).corr()
df_filtro_1 = dfCorr[((dfCorr >= 0.2) | (dfCorr <= -0.2)) & (dfCorr != 1.000)]
df_filtro_1

# Essa linha de código filtra a matriz de correlação dfCorr para exibir apenas as correlações que são maiores ou iguais a 0.3, menores ou iguais a -0.3, e que não são iguais a 1 (evitando a correlação de uma variável com ela mesma).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme()

# Draw a heatmap with the numeric values in each cell
f, ax = plt.subplots(figsize=(16, 6))
sns.heatmap(df_filtro_1, annot=True, linewidths=.5, ax=ax)

In [ ]:
N, P, K, rainfall

In [ ]:
df.columns

## Analise com target

In [ ]:
df.groupby('label').agg({
    'N' : 'median', # -> high variance so its more representative to use median instead of mean
    'P' : 'median', # -> high variance so its more representative to use median instead of mean
    'K' : 'median', # -> high variance so its more representative to use median instead of mean
    'temperature': 'mean',
    'humidity' : 'mean', 
    'ph': 'mean', 
    'rainfall' : 'median' # -> high variance so its more representative to use median instead of mean
})

In [ ]:
df_dummy = df[['label']].copy()

df_dummy = pd.get_dummies(df_dummy)
for c in df_dummy.columns:

    df_dummy[c] = df_dummy[c].apply(lambda x: 1 if x == True else 0)

In [ ]:
df_dummy

In [ ]:
df3 = df.drop('label', axis=1).copy()
df3[df_dummy.columns] = df_dummy


In [ ]:
cols = [c for c in df3.columns if c != 'label_maize']
cols = [c for c in cols if c not in df_dummy]
n_cols = 3
n_rows = -(-len(cols) // n_cols)  # ceil division

fig, axes = plt.subplots(n_rows, n_cols, figsize=(22, n_rows * 5))
axes = axes.flatten()

for i, column in enumerate(cols):
    sns.boxplot(x="label_maize", y=column, data=df3, ax=axes[i])
    axes[i].set_title(f"{column} vs Crop Type")
    axes[i].set_xlabel("")
    axes[i].tick_params(axis='x', rotation=90)

# Esconde subplots vazios (caso o número de colunas não seja múltiplo de n_cols)
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Variáveis vs label_maize", fontsize=16, y=1.01)
plt.tight_layout()
plt.show()




In [ ]:
# Filtrando a matriz de correlação
dfCorr = df3.corr()
df_filtro_1 = dfCorr[((dfCorr >= 0.2) | (dfCorr <= -0.2))]
df_filtro_1

# Essa linha de código filtra a matriz de correlação dfCorr para exibir apenas as correlações que são maiores ou iguais a 0.3, menores ou iguais a -0.3, e que não são iguais a 1 (evitando a correlação de uma variável com ela mesma).

In [ ]:
df_filtro_2 = df_filtro_1.iloc[0:7][df_dummy.columns]

In [ ]:
df_filtro_2

In [ ]:
df_filtro_2.dropna(axis=1, how='all').shape


In [ ]:
df_filtro_2.shape

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme()

# Draw a heatmap with the numeric values in each cell
f, ax = plt.subplots(figsize=(25, 6))
sns.heatmap(df_filtro_2, annot=True, linewidths=.5, ax=ax)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme()

# Draw a heatmap with the numeric values in each cell
f, ax = plt.subplots(figsize=(20, 6))
sns.heatmap(df_filtro_2.dropna(axis=1, how='all'), annot=True, linewidths=.5, ax=ax)

## Analises:

### Relações mais fortes

1. Apple e Grapes com P e K
2. Umidade com Chickpea e Kidneybeans

### Clusters

1. Nitrogenio com Banana, Coffee, Muskmelon, Cotton e Watermelon
2. Temperature com Chickpea, Kidneybeans, mango e Papaya
3. Rainfall com Coconut, Jute, Lentil, Mothbeans, Mungbeans, Rice, Muskmelon e Watermelon
4. P com Coconut, Muskmelon, Orange, Pomegranate e Watermelon


### Variavel sem padrão aparente

1. Pigeonpeas

Vale ressaltar a análise de uma terceira variável 


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Configuração visual padrão
sns.set_theme(style="whitegrid", palette="pastel")

# ==========================================
# 1. Apple e Grapes com P e K
# ==========================================
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
fig.suptitle('Análise de Fósforo (P) e Potássio (K) para Maçã e Uva', fontsize=16)

sns.boxplot(ax=axes[0, 0], data=df3, x="label_apple", y="P")
sns.boxplot(ax=axes[0, 1], data=df3, x="label_apple", y="K")
sns.boxplot(ax=axes[1, 0], data=df3, x="label_grapes", y="P")
sns.boxplot(ax=axes[1, 1], data=df3, x="label_grapes", y="K")

plt.tight_layout()
plt.show()

# ==========================================
# 2. Umidade com Chickpea e Kidneybeans
# ==========================================
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Impacto da Umidade (Humidity)', fontsize=16)

#sns.boxplot(ax=axes[0], data=df3, x="ph", hue="label_chickpea", y="humidity")
#sns.boxplot(ax=axes[1], data=df3, x="ph",hue="label_kidneybeans", y="humidity")

# Cria pontos mostrando a relação pH vs Umidade, coloridos pelo tipo de grão
sns.scatterplot(ax=axes[0], data=df3, x="ph", y="humidity", hue="label_chickpea")
sns.scatterplot(ax=axes[1], data=df3, x="ph", y="humidity", hue="label_kidneybeans")

plt.tight_layout()
plt.show()

# ==========================================
# 3. Nitrogênio (N) com multiplas culturas
# ==========================================
culturas_n = ['label_banana', 'label_coffee', 'label_cotton', 'label_muskmelon', 'label_watermelon']

fig, axes = plt.subplots(1, 5, figsize=(20, 5))
fig.suptitle('Comportamento do Nitrogênio (N)', fontsize=16)

for i, cultura in enumerate(culturas_n):
    sns.boxplot(ax=axes[i], data=df3, x=cultura, y="N")
    axes[i].set_title(cultura.replace('label_', '').capitalize())
    

plt.tight_layout()
plt.show()

# ==========================================
# 4. Temperatura com Chickpea, Kidneybeans, Mango e Papaya
# ==========================================
culturas_temp = ['label_chickpea', 'label_kidneybeans']

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
fig.suptitle('Impacto da Temperatura', fontsize=16)

#for i, cultura in enumerate(culturas_temp):
#    sns.boxplot(ax=axes[i], data=df3, x=cultura, y="temperature")
#    axes[i].set_title(cultura.replace('label_', '').capitalize())

sns.scatterplot(ax=axes[0], data=df3, x="ph", y="temperature", hue="label_chickpea")
sns.scatterplot(ax=axes[1], data=df3, x="ph", y="temperature", hue="label_kidneybeans")

sns.scatterplot(ax=axes[2], data=df3, x="humidity", y="temperature", hue="label_mango")
sns.scatterplot(ax=axes[3], data=df3, x="humidity", y="temperature", hue="label_papaya")

plt.tight_layout()
plt.show()

# ==========================================
# 5. Rainfall com Coconut, Jute, Lentil, Mothbeans, Mungbeans, Rice, Muskmelon e Watermelon
# ==========================================

columns_rainfall = ['label_coconut', 'label_jute', 'label_lentil', 'label_mothbeans', 'label_mungbean', 'label_rice', 'label_muskmelon', 'label_watermelon']
fig, axes = plt.subplots(1, 8, figsize=(20, 5))
fig.suptitle('Comportamento do Rainfall', fontsize=16)

for i, cultura in enumerate(columns_rainfall):
    sns.boxplot(ax=axes[i], data=df3, x=cultura, y="rainfall")
    axes[i].set_title(cultura.replace('label_', '').capitalize())
    

plt.tight_layout()
plt.show()


# ==========================================
# 6. P com Coconut, Muskmelon, Orange, Pomegranate e Watermelon
# ==========================================

columns_p = ['label_coconut', 'label_orange', 'label_pomegranate', 'label_muskmelon', 'label_watermelon']
fig, axes = plt.subplots(1, 8, figsize=(20, 5))
fig.suptitle('Comportamento do P (Fósforo)', fontsize=16)

for i, cultura in enumerate(columns_p):
    sns.boxplot(ax=axes[i], data=df3, x=cultura, y="P")
    axes[i].set_title(cultura.replace('label_', '').capitalize())
    

plt.tight_layout()
plt.show()

# ==========================================
# 7. Variável com uma correlação
# ==========================================

plt.figure(figsize=(10, 5))


sns.boxplot(data=df3, x='label_pigeonpeas', y='humidity')

plt.title('Comportamento do Pigeonpeas com Umidade', fontsize=16)

plt.tight_layout()
plt.show()



Fósforo (P) e Potássio (K) para Maçã e Uva:

    O modelo define a recomendação de maçã com base em níveis de fósforo que se situam mais altos, em torno de 130 a 140. Quando a classe alvo não é maçã (valor zero), os níveis de fósforo encontrados estão mais baixos, aproximadamente entre 30 e 64. A correlação para maçã é fortemente positiva tanto para o fósforo (+0.52) quanto para o potássio (+0.65). Ou seja, quanto maiores os níveis desses dois nutrientes, maior a chance de o modelo recomendar o plantio de maçã.
    O mesmo padrão se aplica à uva, que possui valores de correlação positivos extremamente parecidos. Quando os níveis de P e K estão baixos, o modelo recusa ambas as culturas. Devido a essa similaridade na correlação linear, a distinção exata entre a escolha de maçã ou uva ocorre por meio de fatores não lineares ou variáveis terceiras.

Umidade e pH para Grão-de-bico (Chickpea) e Feijão (Kidney beans):

    Essas duas culturas possuem correlações negativas altas e muito similares com a variável umidade (Feijão com -0.49 e Grão-de-bico com -0.54). Para umidades baixas, o modelo passa a recomendar o plantio: o feijão suporta faixas de 20 a 25, enquanto o grão-de-bico suporta valores levemente inferiores, na faixa de 17 a 19. A recusa passa a ocorrer em faixas de alta umidade: quando a umidade oscila entre 65 e 90, o modelo não recomenda essas culturas.
    O fator determinante que diferencia e separa o grão-de-bico do feijão é o pH. O grão-de-bico possui uma correlação positiva com o pH, preferindo solos mais alcalinos (pH mais alto). Já o feijão tem uma correlação negativa, preferindo níveis mais baixos (solos mais ácidos). Quando o pH ultrapassa a marca de 6 a 9 (considerado alto na análise), o modelo recomenda o grão-de-bico e não recomenda o feijão.

Nitrogênio (N) para Banana, Café, Algodão, Melão e Melancia:

    Existe uma forte similaridade no comportamento dessas classes em relação ao Nitrogênio. Todas apresentam correlações positivas que variam entre +0.29 e +0.40, sendo a do algodão a mais acentuada (+0.40). De forma geral, níveis baixos de nitrogênio resultam na recusa do modelo para o plantio dessas culturas (ou seja, elas exigem níveis mais altos). O algodão destaca-se como a cultura em que o modelo tem mais facilidade para definir a escolha ou recusa apenas avaliando a forte correlação linear positiva.

Temperatura para Manga e Papaya:

    Ao observar a temperatura e a umidade, nota-se um comportamento inversamente proporcional na correlação para essas duas culturas. A manga apresenta uma correlação negativa com a temperatura, o que indica que valores de temperatura mais baixos aumentam as chances da sua recomendação. Em contraste, o papaya possui uma correlação positiva, sendo preferido pelo modelo em cenários onde a temperatura é comparativamente mais alta.

Rainfall com Coconut, Jute, Lentil, Mothbeans, Mungbeans, Rice, Muskmelon e Watermelon

    Coconut tende a preferir maiores níveis de chuva (150-200mm), porém há indicios de os dados tenderem a recusar a plantação de coconut em valores altos. Isso deve-se a influência de uma variável terceira ou influência não-linear. Uma alternativa seria baixas temperaturas para coconut ou alto nível de potassio.
    Jute apresenta comportamento semelhante a Coconut, preferido níveis entre 160-180mm, recusando a partir de valor de 70-120mm. É uma planta que necessita de bastante chuva. Há casos em que recusa com altas chuvas, sugerindo influência terceira. Em baixas temperaturas isso pode acabar acontecendo ou há uma influência não-linear.
    Lentil é uma planta resistente a secas, crescendo em ambientes de baixa chuva, o que explica os dados tenderem a escolher Lentil quando a faixa de mm está menor de 50 e acima de 40mm. Como esperado valores maiores de chuva afetam a escolha de lentil.
    Mothbeans é uma leguminosa igual a lentil, que também é resistente a secas. Ela em questão aceita níveis ligeiramente maiores de chuva (40-55mm), demonstrando comportamento semelhante a Lentil quando há alta chuva. Há outliers assim como nos outros na questão de muita chuva. Muita chuva é esperado ter um comportamento não-linear.
    Mungbean é uma leguminosa resistente a seca também, com os dados tendendo a escolher nos mesmos níveis que as outras leguminosas. Comportamento esperado.
    Rice é um cereal semi-aquatico, crescendo em ambiente agalados e com bastante água. Os dados tendem a escolher rice com chuvas acima de 200mm com a maior concentração até 260mm. Há valores indicando recusa (outliers) mesmo com altos níveis de chuva, isso pode ter influência te outros nutrientes
    Muskmelon e Watermelon são frutas tropicais intolerantes a encharcamento, preferindo climas mais secos na sus fase final. Ambas necessitam solos bem drenados, o que explica os dados tenderem a escolher ambas com níveis baixos de chuva (valores de 20mm até 60mm). Por conta da correlação linear apresentar valores parecidos para ambas, a separação provavelmente se deve a fator não-lineares nos dados. Não é possível distinguir com apenas essa analise

 P com Coconut, Muskmelon, Orange, Pomegranate e Watermelon
    
    Todas as plantas apresentam baixos níveis de Fósforo quando os dados tendem a escolher elas, por serem plantas com maior tolerância a baixos níveis de fósforo no longo-prazo. Isso não indica que elas não precisem, e sim indicam que, de acordo com as condições do solo atualmente, essas plantas dará o maior retorno ao agricultor. Isso minimiza o custo do agricultor com fertilizantes. 
    Elas também indicam níveis de correlações parecidos a outras variáveis, como humidity, mas nada que seja possível separa-las. 

Pigeonspeas com umiodade
    
    É uma planta resistente a secas devido as suas raizes profundas, o que explica sua correlação negativa com umidade. No entanto até sua fase adulta é necessário umidade adequada no solo para garantir germinação saudavel

Maize

    Analisando os gráficos não é possível pegar nenhuma correlação linear com as outras variáveis





In [ ]:
df2 = df.copy()
df2['label'] = df2['label'].astype('category')

decode_map = dict(enumerate(df2['label'].cat.categories))

df2['label'] = df2['label'].cat.codes

In [ ]:
df2 = df.copy()
df2['label'] = df2['label'].astype('category')

decode_map = dict(enumerate(df2['label'].cat.categories))

df2['label'] = df2['label'].cat.codes

X = df2.drop('label', axis=1)
y = df2['label']

Vamos usar agora valores SHAP para analisar a interação entre variaveis e a variavel target.

Para não errarmos e termos certeza que o modelo utilizado vai ser realista diante da nossa analise, vamos utilizar validação cruzada para verificar se o modelo serve ou não. Essa medida vai servir como confirmação de que as informações que vamos obter importam e que não estamos vendo ruídos

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestClassifier

clf = RandomForestClassifier()
scores = cross_val_score(clf, X, y, cv=5, scoring='f1_macro') # vamos deixar o cross_val_score separar em folds nativamente pois não há necessidade de balancear as classes
print(f'Score -> {np.mean(scores)}')

In [ ]:
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split

# Train a machine learning model}
from sklearn.ensemble import RandomForestClassifier
clf = RandomForestClassifier()
clf.fit(X, y)


In [ ]:
import shap

# 1. Initialize Explainer with probability output
# For Random Forest, TreeExplainer is fastest and highly accurate
explainer = shap.TreeExplainer(clf, X, model_output="probability")
shap_values = explainer(X)

# In SHAP, the output dimensions are: [samples, features, classes]
print(shap_values.shape) 




In [ ]:

# Defina o tamanho que desejar na tupla plot_size (ex: 12 de largura, 6 de altura)
shap.summary_plot(shap_values, X, plot_type="bar", plot_size=(30, 10), class_names=decode_map)




## Então o que define a escolha de uma plantação especifica?